# Gender Wage Gap in the US, 1976-2025
CPS ASEC from IPUMS | Course research project

**Question.** How much has the gender wage gap closed in the US since 1976, and for whom?

**Rule:** every table and chart uses the survey weight, `asecwt`. Unweighted counts are only for checking data.

**Table first, chart second.** Print the weighted table (1 decimal), then chart exactly that table. Bar and line charts start at 0.

---
## 0. Setup
The file `jobs_assignment_data_20261104.csv` is in the root of My Drive: income years 1976-2025, ages 16-64, wage and salary workers with positive wages. The file has 85 columns and is 1.6 GB, so we load only the 18 below. `year` is the **income year** (the year the wages were earned). The setup cell adds a text label for each coded variable (`female_label`, `fulltime_label`, `educ5_label`, and so on).

| Variable | Meaning |
|---|---|
| `year`, `age`, `asecwt` | income year; age 16-64; weight (Americans each row stands for) |
| `female` | 1 = female, 0 = male |
| `fulltime`, `hrswk`, `wkswork` | full-time (1) or part-time (0) last year, full-time = 35+ usual weekly hours; usual weekly hours; weeks worked last year |
| `incwage_real`, `hrwage_real` | yearly wage and salary income; hourly wage (yearly wage / (weeks x hours)); both in 2025 dollars |
| `agegrp` | 16-24, 25-34, 35-44, 45-54, 55-64 |
| `educ5` | None/preschool, Less than HS, High school, Some college, Bachelor's+ |
| `raceeth` | White, Black (non-Hispanic), Hispanic (any race), Other; missing = don't know or no response |
| `marst3` | Married, Previously married, Never married |
| `nchild_cat`, `child_u5` | number of children (0, 1, 2, 3+); 1 = child under 5 |
| `occ6` | 6 occupation groups; missing = not in universe, military, 999, or other |
| `classwkr3` | Private, Government (the self-employed were removed) |
| `metro` | 1 = in a metro area, 0 = not; missing in some years |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount("/content/drive")

data_path = "/content/drive/My Drive/jobs_assignment_data_20261104.csv"        # change here if the file moves
cols = ["year", "age", "asecwt", "female", "fulltime", "hrswk", "wkswork", "incwage_real", "hrwage_real",
        "agegrp", "educ5", "raceeth", "marst3", "nchild_cat", "child_u5", "occ6", "classwkr3", "metro"]

df = pd.read_csv(data_path, usecols=cols)                                      # read only these columns: the file is big
print(df.shape)                                                                # rows, columns we loaded
print(list(df.columns))                                                        # their names
print(len(pd.read_csv(data_path, nrows=0).columns), "columns in the file")     # nrows=0 reads only the header

In [ ]:
# Labels for the codes, then a check: every code should land in exactly one label
df["female_label"] = df["female"].map({0: "Male", 1: "Female"})
df["fulltime_label"] = df["fulltime"].map({0: "Part-time", 1: "Full-time"})
df["agegrp_label"] = df["agegrp"].map({1: "16-24", 2: "25-34", 3: "35-44", 4: "45-54", 5: "55-64"})
df["educ5_label"] = df["educ5"].map({1: "None/preschool", 2: "Less than HS", 3: "High school", 4: "Some college", 5: "Bachelor's+"})
df["raceeth_label"] = df["raceeth"].map({1: "Hispanic", 2: "White", 3: "Black", 4: "Other"})        # missing stays missing
df["marst3_label"] = df["marst3"].map({1: "Married", 2: "Previously married", 3: "Never married"})
df["nchild_cat_label"] = df["nchild_cat"].map({0: "0", 1: "1", 2: "2", 3: "3+"})
df["child_u5_label"] = df["child_u5"].map({0: "No child under 5", 1: "Child under 5"})
df["occ6_label"] = df["occ6"].map({1: "Managers and professionals", 2: "Technical, sales and admin", 3: "Service",
                                   4: "Farming, forestry and fishing", 5: "Craft and repair", 6: "Operators and laborers"})   # missing stays missing
df["classwkr3_label"] = df["classwkr3"].map({1: "Private", 2: "Government"})
df["metro_label"] = df["metro"].map({0: "Non-metro", 1: "Metro"})                                  # missing stays missing
for name in ["female", "fulltime", "agegrp", "educ5", "raceeth", "marst3", "nchild_cat", "child_u5", "occ6", "classwkr3", "metro"]:
    print(pd.crosstab(df[name], df[name + "_label"], dropna=False))                                # each row should have one non-zero cell

In [ ]:
# Weighted mean: multiply each value by its weight, add up, divide by the sum of the weights
def wmean(data, var):                                                   # a table and a column name
    ok = data[var].notna()                                              # rows that have this column
    return np.average(data.loc[ok, var], weights=data.loc[ok, "asecwt"])   # the weighted mean

---
## 1. Summary table
**Question.** What does each loaded column look like before any math? The mean is weighted with `asecwt`. The count (n), the missing count, the min and the max are unweighted: they are data checks. A mean of a 0/1 column like `female` is a share. A mean of a code with several groups like `educ5` means nothing by itself; we keep the row to check n, missing, min and max.

In [ ]:
summary = pd.DataFrame({
    "n": df[cols].count(),                                  # people with a value
    "missing": df[cols].isna().sum(),                       # people without a value
    "weighted mean": [wmean(df, c) for c in cols],          # mean weighted with asecwt
    "min": df[cols].min(),                                  # smallest value
    "max": df[cols].max(),                                  # largest value
}, index=cols)
print(summary.round(1))